# 57 · Mở băng block 8–11 **với kiến trúc đầy đủ**

## Câu hỏi

Đòn bẩy lớn nhất từng đo là fine-tune: `04b` LP `0.3191` → FT `0.4201` = **+0.101**. Ta đã ăn phần
đó ở `38a` — nhưng backbone được fine-tune với **đầu đọc đơn giản**, rồi đông cứng, và toàn bộ
`Tier1` / `Tier1Img` / ML-Decoder học **trên đặc trưng chết**.

```
38a  fine-tune voi dau doc DON GIAN roi dong cung -> dac trung toi uu cho DAU DOC CU
ta   doc bang ML-Decoder + Tier1 + Tier1Img       -> chua bao gio duoc thich nghi voi cach doc nay
```

Riêng nhánh **phim trước** là thứ `38a` hoàn toàn không biết: đặc trưng chưa từng bị ép mã hoá
*"cái gì thay đổi giữa hai phim"*.

**Phản biện đã nêu** *(người dùng)*: MIMIC nằm trong corpus pretrain của RAD-DINO, mở ra chắc vô
nghĩa. Đúng về **quen miền** (khoảng cách miền ≈ 0), nhưng **thẳng hàng với nhiệm vụ** là chuyện
khác — mục tiêu DINO là bất biến augmentation, không có khái niệm tách `Lobar Atelectasis` khỏi
`Atelectasis`. `04b` đo `+0.101` **trên chính dataset RAD-DINO đã thấy**. Phần lớn khoản đó ta đã
ăn; cái còn lại hẹp hơn nhiều và **chưa ai đo**.

## Giả thuyết phụ: mở băng có dập được cú sụp của `Tier1Img` không

`Tier1Img` đã làm hỏng arm ở `53` (`imgz`) và ở `56` (**`tpb`**, mốc của cả dự án — tail 0.0685 so
0.1077, `pr_norm` 16.30 so 11.50). Nếu nguyên nhân là *nhánh phim trước phải tự xoay xở trên đặc
trưng tĩnh*, thì cho backbone thích nghi có thể dập luôn cú sụp. Đọc `pr_norm` của hai arm cạnh
nhau mỗi epoch sẽ thấy.

## Ba arm

```
mldec     goc toa do, backbone DONG CUNG                    | upper CHUNG
tpb       KIEN TRUC CHINH, backbone DONG CUNG               | upper CHUNG
tpb_ft    block 8-11 MO BANG (ban sao rieng), LR/10         | upper RIENG, co graph

G_ft = tpb_ft - tpb        <- CAU HOI CHINH
```

`tpb_ft` giữ **bản sao riêng** của block 8–11: nếu dùng chung thì việc cập nhật chúng sẽ thay đổi
đặc trưng của `mldec` và `tpb`, phá luôn hai đối chứng.

## Sửa hai lỗi hiệu năng của `56`

`56` chạy 15 img/s (4.1 giờ/epoch) vì `upper()` không bọc `autocast`, và ba arm ghi là "fwd CHUNG"
thật ra **tính lại riêng từng arm**. Ở đây phần đông cứng tính **một lần** cho cả batch và mọi thứ
chạy bf16.

## C0 · Config

In [ ]:
import os, json, math, time, tarfile, io as _io, re, datetime, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, ('khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`. '
                      'Nhanh du phong "./planB_out" cu bao SAI nguyen nhan.')
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r3'      # doi vi Tier1Img da doi (LayerScale) — DUNG resume qua tag cu
RES57 = os.path.join(OUT_DIR, 'stage4_nb57', RUN_TAG); CKPT = os.path.join(RES57, 'ckpt')
P_OOF = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
# TACH theo notebook: hai tien trinh cung `append` mot file qua Drive FUSE se nuot dong
# cua nhau. 53 va 57 chay song song.
P_TOUCH = os.path.join(OUT_DIR, '_test_touches_nb57.jsonl')
def ensure_res():
    '''Drive FUSE danh roi thu muc RONG — goi TRUOC moi cum ghi (43/38c/38d/41a/41c da dinh).'''
    os.makedirs(RES57, exist_ok=True); os.makedirs(CKPT, exist_ok=True); return RES57
ensure_res()
assert os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR), f'{OUT_DIR} khong doc duoc -> mount hong'
json.dump({'run_tag': RUN_TAG, 'notebook': '57'}, open(os.path.join(RES57, '_run_info.json'), 'w'))
LOCAL = '/content/cxrlt_518'
READY_X = os.path.join(OUT_DIR, 'cxrlt_518_ready')   # kho do 41c C3-bis tao

FT_BLOCKS = (8, 9, 10, 11)      # mo bang 4 block tren. `11`: I_location ~ 0, nen chon theo chi phi.
LR_FT_DIV = 10.0                # LR cho block mo bang = LR/10 — chuan LP-FT, tranh quen pretrain
BB_RUN = 'a2_E3'; SEEDS = (42, 43)
SEED, SIZE, EPOCHS = 42, 518, 10
BATCH, ACCUM = 32, 8                      # EFF 256 — GIU y het 53/56 de so truc tiep
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
NW = min(16, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
DO_TEST, N_TEST_EP = True, 3

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

BB_SRC = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
for p, hd in ((BB_SRC, '38a'), (P_MICRO, '29c'), (P_VOCAB, '29c'), (P_OOF, '52a')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL       # CUNG cach cat nhu 35/37/38*/44*/45/52*/53/56
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,} | '
      f'test {len(I_TE):,}')
print(f'mo bang block {FT_BLOCKS} (LR/{LR_FT_DIV:.0f}) | E={EPOCHS} | '
      f'BATCH {BATCH}x{ACCUM}={BATCH*ACCUM} | seeds {SEEDS}')

## C1 · EHR + túi vi khuẩn + XGB + phim trước — verbatim `53`/`56`

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = np.nan_to_num((E_DF.set_index('study_id')[COLS].reindex(_sid)
                    .apply(pd.to_numeric, errors='coerce').to_numpy('float32')), nan=0.0)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468 -> softmax bao hoa -> NaN
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
COLS = [COLS[i] for i in _keep]; EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all()
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT = np.concatenate([EHR_COL, XGB_F], 1)      # `both` = 217 cot EHR + 40 cot XGB (53: tot nhat)
print(f'FEAT {FEAT.shape} = {EHR_COL.shape[1]} EHR + {C} XGB | tui vi khuan V={V_MIC-1} dai {MIC_L}')

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5 and (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all()

## C2 · Ảnh + backbone + **cổng cắt đôi forward**

`Dinov2Layer` trả về **Tensor trần** ở bản `transformers` này, không phải tuple. `layer(h)[0]` sẽ
lấy **mẫu đầu của batch** và broadcasting che lấp lỗi nên không báo gì — `39` đã dính đúng chỗ này.
`_out()` kiểm `isinstance` trước khi lấy `[0]`.

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T
import shutil

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
# Uu tien KHO da resize san (41c C3-bis tao): copy + bung, vai phut. Neu chua co thi stage tu shard
# goc 1022px — cham hon ~10 lan vi phai giai ma + resize 377k anh.
if _n < len(ORDER) * 0.999 and os.path.isdir(READY_X):
    for f in sorted(x for x in os.listdir(READY_X) if x.endswith('.tar')):
        src = os.path.join(READY_X, f); tmp = os.path.join('/content', f)
        try:
            shutil.copy(src, tmp)                        # copy TRUOC — khong mo tar tren mount
            assert os.path.getsize(tmp) == os.path.getsize(src), 'copy NGAN tu Drive'
            with tarfile.open(tmp) as tar: tar.extractall(LOCAL, filter='data')
        except Exception as e:
            print(f'  {f} HONG ({type(e).__name__}) -> bo qua')
        if os.path.exists(tmp): os.remove(tmp)
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
    print(f'hydrate tu kho: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
    assert os.path.isdir(SHARD_DIR), f'khong thay CA {READY_X} lan {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        try:
            im = Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC)
        except Exception:
            return                                       # mot anh hong KHONG duoc giet ca lan chay
        tmp = dst + '.tmp'; im.save(tmp, 'JPEG', quality=95); os.replace(tmp, dst)   # NGUYEN TU
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()     # doc BULK; lap tren mount FUSE se TREO
            assert len(buf) == os.path.getsize(tp), f'{os.path.basename(tp)}: doc NGAN tren mount'
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'
print(f'anh local: {_n:,} / {len(ORDER):,}')

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_sd0 = {k: v.clone() for k, v in AutoModel.from_pretrained('microsoft/rad-dino').state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
del _sd0
assert _d > 1e-6, f'backbone GIONG HET pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
LAYERS = backbone.encoder.layer; NL = len(LAYERS)
GRAD_FROM = min(FT_BLOCKS)
assert max(FT_BLOCKS) < NL and list(FT_BLOCKS) == list(range(GRAD_FROM, NL)), (
    f'FT_BLOCKS {FT_BLOCKS} phai la day LIEN TUC den block cuoi ({NL-1}) — neu khong thi phan '
    'dong cung o TREN chung khong dung chung duoc voi arm ft.')
_cfg = backbone.config
_dp = (getattr(_cfg, 'attention_probs_dropout_prob', 0), getattr(_cfg, 'hidden_dropout_prob', 0),
       getattr(_cfg, 'drop_path_rate', 0))
print(f'BACKBONE `ft` 38a/{BB_RUN} | {NL} block | d={D_CXR} | lech pretrain {_d:.3f}')
print(f'dropout (attn, hidden, drop_path) = {_dp}'
      + ('  -> bang 0, che do train/eval khong anh huong' if max(_dp) == 0 else
         '  -> KHAC 0: ban sao PHAI chay .eval() de khop ban goc'))

def _out(r):
    '''Tensor TRAN, khong phai tuple: `r[0]` se lay MAU DAU cua batch va broadcasting che lap loi.'''
    return r[0] if isinstance(r, tuple) else r

class DS(Dataset):
    '''Tra ve CA HAI anh: hien tai va PHIM TRUOC (+ `has`). `ehr_perm` cho TRAO NHAN QUA.'''
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return s.tf(Image.merge('RGB', (im, im, im)))
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if pr >= 0 else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), pos=k,
                    y=torch.from_numpy(YM[i]), row=i,
                    erow=(i if s.perm is None else int(s.perm[k])))

def loader(idx, tf, shuffle, bs, perm=None, nw=NW, drop_last=None):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

def run_blocks(h, lo, hi, layers=None):
    L = LAYERS if layers is None else layers
    for i in range(lo, hi): h = _out(L[i - (0 if layers is None else lo)](h))
    return h

# --- cong 1: forward CAT DOI phai bang forward NGUYEN VEN ---
_b0 = next(iter(loader(I_VA[:4], eval_tf, False, 4, nw=0))); _x = _b0['image'].to(dev)
with torch.no_grad():
    _full = backbone(pixel_values=_x).last_hidden_state
    _split = backbone.layernorm(run_blocks(run_blocks(backbone.embeddings(_x), 0, GRAD_FROM),
                                           GRAD_FROM, NL))
_e1 = float((_full - _split).abs().max())
assert _e1 < 1e-3, (f'forward CAT DOI lech {_e1:.2e} -> da cat SAI cho trong kien truc HF. '
                    'DUNG chay tiep, moi so sau day vo nghia.')
N_TOK = _full.shape[1] - 1
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2
print(f'cong 1: forward cat doi khop toi {_e1:.1e} | {N_TOK} patch token')

## C3 · Kiến trúc — verbatim `53`/`56`

In [ ]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    '''`const=True`: MOT chi so, khong phai MIC_L ban sao (tui that TB 1.7 vi khuan).'''
    def __init__(s, V, d, const=False):
        super().__init__()
        s.const = const; s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w):
        if s.const:
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    '''d_in token COT + 1 TONG HOP + 1 BAG. Chuoi nay CHUA k1 nhu truong hop rieng.'''
    def __init__(s, d_in, d=256, const=False, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_in + 1, d) * 0.02)
        s.mb = MicroBag(V, d, const) if V else None
        if s.mb is not None:
            s.bln = nn.LayerNorm(d); s.cln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, e, mids=None, mcnt=None):
        if s.const: base = s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        else:
            col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
            base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([s.cln(base), s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    '''Q = 1369 PATCH, K/V = token EHR. `pr` zero-init; `ln_a` chuan hoa DAU VAO cua phep chieu
       (38d: thieu no thi arm doi chung sup 0.2416 -> 0.0681).'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Phim truoc o muc PATCH, chay o khong gian D_CXR (TRUOC `pin`).
       KHONG tru thang `patch - prior`: hai phim khong duoc dang ky (tu the, do hit vao). '''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        # LAYERSCALE. `pr` zero-init cho dong nhat thuc o buoc 0 nhung KHONG chan duoc toc do
        # lon len: AdamW cap nhat ~lr/buoc/tham so BAT KE gradient nho the nao, nen mot nhanh xuat
        # phat tu 0 lon TUYEN TINH theo buoc x LR. Do `ln_a` chuan hoa `patch - a` (thuong gan 0,
        # phan lon la nhieu) ve chuan don vi, bien do dau ra do TRON VEN `‖pr‖` quyet dinh. Khi no
        # at `patch` thi ML-Decoder doi che do dot ngot -> sup. Da xay ra 3 lan, luon o ep2-4 tuc
        # ngay sau dinh warmup (0.05 x E = nua epoch 1), va `clip` do 0.0% vi gradient VAN binh
        # thuong — van de o BUOC DI, khong o gradient.
        # Nay `pr` khoi tao CHUAN va mot he so LayerScale `1e-4` giu phan du gan 0 luc dau; ca hai
        # deu co gradient that, va he so phai HOC de lon len thay vi lon lên vi quan tinh AdamW.
        s.pr = nn.Linear(d, d)
        s.gamma = nn.Parameter(torch.full((d,), 1e-4))
        s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.gamma * s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)      # khong co phim truoc -> mask nhanh di han

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = patch, co the da bi Tier1 dieu bien.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, w = s.img(q, kv, need_w); return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45/52*/53/56. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

## C4 · Ba arm + bản sao block 8–11 + cổng khoá trước

In [ ]:
ARMS = {
  'mldec':  dict(ehr=False, img=False, ft=False),
  'tpb':    dict(ehr=True,  img=True,  ft=False),
  'tpb_ft': dict(ehr=True,  img=True,  ft=True),
}
FT_ARMS = [t for t, c in ARMS.items() if c['ft']]

def build_arms(seed=SEED):
    M, E, I, U = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)     # gieo lai TRUOC TUNG arm -> khoi tao giong het nhau
        if cfg['ehr']: E[tag] = ColTokBag(FEAT.shape[1], D_ETOK, False, V=V_MIC)
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=cfg['ehr'])
        if cfg['img']: I[tag] = Tier1Img(D_CXR)
        if cfg['ft']:
            # BAN SAO RIENG cua block 8-11 + layernorm. Dung chung thi viec cap nhat chung se
            # doi dac trung cua `mldec` va `tpb` -> pha luon hai doi chung.
            U[tag] = nn.ModuleList([copy.deepcopy(LAYERS[i]) for i in FT_BLOCKS]
                                   + [copy.deepcopy(backbone.layernorm)])
            for p in U[tag].parameters(): p.requires_grad_(True)
    return M.to(dev), E.to(dev), I.to(dev), U.to(dev)

MOD, ETK, IMG, UPF = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + [x[t] for x in (ETK, IMG, UPF) if t in x])
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<12} {"tham so":>12} {"trong do block 8-11":>21}')
for t in ARMS:
    nu = sum(p.numel() for p in UPF[t].parameters()) if t in UPF else 0
    print(f'  {t:<10} {NP_[t]:>12,} {nu:>21,}')

_e = torch.randn(4, FEAT.shape[1], device=dev)
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
_p = torch.randn(4, N_TOK, D_CXR, device=dev); _pp = torch.randn(4, N_TOK, D_CXR, device=dev)
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)
tok = lambda t: ETK[t](_e, _mi, _mc)

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone GOC phai dong cung'
assert not MOD['mldec'].q.weight.requires_grad, 'query_embed dang hoc'
with torch.no_grad():
    m = MOD['tpb'].eval(); l1, _ = m(_p, tok('tpb'))
    _k = m.t1; m.t1 = None; l2, _ = m(_p, tok('tpb')); m.t1 = _k
    assert (l1 - l2).abs().max() < 1e-5, 'Tier1 KHONG zero-init'
    for t in IMG:
        # Cong TUONG DOI thay vi tuyet doi: LayerScale cho dong nhat thuc XAP XI (~1e-4), khong
        # bang chinh xac. Va ti le bien do moi la thu ta thuc su quan tam — chinh no vuot 1 la sup.
        _rel = float((IMG[t](_p, _pp, _h1) - _p).norm() / (_p.norm() + 1e-9))
        assert _rel < 1e-3, f'{t}: Tier1Img nhieu {_rel:.2e} luc khoi tao, phai < 1e-3'
print('cong 2: Tier1 va Tier1Img deu zero-init OK')

# --- cong 3: ban sao block 8-11 phai KHOI DAU y het ban goc, va DOC LAP voi no ---
_x4 = torch.randn(4, N_TOK + 1, D_CXR, device=dev)
with torch.no_grad():
    _a = backbone.layernorm(run_blocks(_x4, GRAD_FROM, NL))
    _h = _x4
    for k, i in enumerate(FT_BLOCKS): _h = _out(UPF['tpb_ft'][k](_h))
    _b2 = UPF['tpb_ft'][-1](_h)
    _de = float((_a - _b2).abs().max())
assert _de < 1e-4, f'ban sao lech ban goc {_de:.2e} ngay tu dau -> deepcopy SAI'
# KHONG bam vao ten thuoc tinh noi bo cua HF (`.attention.attention.query`) — chinh lop gia dinh
# do da lam hong `39`. Lay THAM SO DAU TIEN cua block, dung voi moi kien truc.
_f2 = lambda m: next(p for p in m.parameters() if p.ndim > 1)
_pf = _f2(UPF['tpb_ft'][0]); _pg = _f2(LAYERS[GRAD_FROM])
_before = _pg.detach().clone()
with torch.no_grad(): _pf.add_(1.0)
_moved = float((_pg - _before).abs().max())
with torch.no_grad(): _pf.sub_(1.0)
assert _moved == 0.0, ('sua ban sao lam DOI ban goc -> deepcopy chia se bo nho, `mldec`/`tpb` se bi '
                       'keo theo va hai doi chung thanh vo nghia')
print(f'cong 3: ban sao khop ban goc toi {_de:.1e} va DOC LAP bo nho OK')
with torch.no_grad():
    _ti = IMG['tpb']; _ti.pr.weight.normal_(0, 0.02)
    _r = float((_ti(_p, _pp*100.0, _h1) - _p).abs().max() / ((_ti(_p, _pp, _h1) - _p).abs().max()+1e-9))
    assert (_ti(_p, _pp, _h0) - _p).abs().max() < 1e-6, 'has=0 KHONG cat duoc nhanh phim truoc'
    nn.init.zeros_(_ti.pr.weight)
assert _r < 3.0, f'phong dai K/V 100x -> bien do ra tang {_r:.1f}x, `ln_a` khong chan duoc'
print(f'cong 4: `ln_a` chan bien do OK ({_r:.2f}x) | has=0 cat sach nhanh OK')
MOD, ETK, IMG, UPF = build_arms()

## C5 · Chấm điểm

In [ ]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
_rs6 = np.random.default_rng(6); _yy = (_rs6.random((3000, C)) < 0.2).astype('float32')
_ss6 = _rs6.random((3000, C)).astype('float32')
_dd6 = float(np.nanmax(np.abs(ap_cols(_yy, _ss6) -
        np.array([average_precision_score(_yy[:, j], _ss6[:, j]) for j in range(C)]))))
assert _dd6 < 1e-6, f'ap_cols lech sklearn {_dd6:.2e}'
print(f'cong 5: ap_cols khop sklearn toi {_dd6:.2e}')

## C6 · Huấn luyện

**Sửa hai lỗi hiệu năng của `56`.** Phần đông cứng (block 8–11 gốc) tính **một lần** cho cả batch
rồi ba arm dùng chung; mọi thứ chạy dưới `autocast`. `56` gọi `upper()` riêng cho từng arm và chạy
fp32 ⇒ 15 img/s. Kỳ vọng ở đây nhanh hơn ~5 lần.

Ghi mỗi epoch: `pr_norm` (‖`Tier1Img.pr`‖ — theo dõi cú sụp), `clip_pct`, và **`drift`** = mức
block 8–11 của arm `ft` đã đi xa bản gốc bao nhiêu.

In [ ]:
crit = ASL(gn=4, gp=0).to(dev)
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = torch.from_numpy(FEAT)
# Lay tham so 2-D dau tien, KHONG phai `next(iter(...))`: tham so dau cua mot `Dinov2Layer` la
# `norm1.weight` (1-D), ma nhom ndim<=1 von da co weight_decay=0 -> do o do thi MU voi chinh thu
# ta muon theo doi. Weight decay chi tac dong len ma tran 2-D.
# Van KHONG bam vao ten thuoc tinh cua HF (`.attention.attention.query`) — gia dinh do da lam hong `39`.
_first2d = lambda m: next(p for p in m.parameters() if p.ndim > 1)
_REF = {i: _first2d(LAYERS[i]).detach().clone() for i in FT_BLOCKS}

@torch.no_grad()
def shared_lo(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return run_blocks(backbone.embeddings(img), 0, GRAD_FROM)

@torch.no_grad()
def upper_frozen(h):
    '''Block 8-11 GOC. Tinh MOT lan cho ca batch — moi arm dong cung dung chung.'''
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return backbone.layernorm(run_blocks(h, GRAD_FROM, NL))[:, 1:].float()

def upper_ft(h, t):
    '''Block 8-11 BAN SAO cua arm `t`, CO graph.'''
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        for k in range(len(FT_BLOCKS)): h = _out(UPF[t][k](h))
        return UPF[t][-1](h)[:, 1:].float()

def arm_forward(t, pf, ppf, hl, hlp, erow, has):
    '''`pf`/`ppf` = dac trung DONG CUNG dung chung. Arm `ft` bo qua chung, tu tinh tu `hl`.'''
    cfg = ARMS[t]
    et = ETK[t](FEAT_T[erow].to(dev), MIDS_T[erow].to(dev), MCNT_T[erow].to(dev)) \
         if cfg['ehr'] else None
    if cfg['ft']: p, pp = upper_ft(hl, t), upper_ft(hlp, t)
    else:         p, pp = pf, ppf
    if cfg['img']: p = IMG[t](p, pp, has)
    return MOD[t](p, et)[0]

@torch.no_grad()
def predict(idx, perm=None, bs=32):
    MOD.eval(); ETK.eval(); IMG.eval(); UPF.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(loader(idx, eval_tf, False, bs, perm, nw=0, drop_last=False),
                  desc='eval', leave=False):
        hl = shared_lo(b['image'].to(dev, non_blocking=True))
        hlp = shared_lo(b['image_prev'].to(dev, non_blocking=True)); hs = b['has'].to(dev)
        pf, ppf = upper_frozen(hl), upper_frozen(hlp)
        for t in ARMS:
            P[t][b['pos'].numpy()] = torch.sigmoid(
                arm_forward(t, pf, ppf, hl, hlp, b['erow'], hs).float()).cpu().numpy()
    assert not any(np.isnan(v).any() for v in P.values()), 'predict cho ra NaN'
    return P

P_LOG = os.path.join(RES57, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL = loader(I_TF, train_tf, True, BATCH)

for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    MOD, ETK, IMG, UPF = build_arms(seed)
    PARAM_OF = {t: [p for m in ([MOD[t]] + [x[t] for x in (ETK, IMG, UPF) if t in x])
                    for p in m.parameters() if p.requires_grad] for t in ARMS}
    N_CLIP = {t: 0 for t in ARMS}; N_SKIP = {t: 0 for t in ARMS}; N_STEP = 0
    # Block mo bang dung LR/10 (chuan LP-FT): LR day du se xoa kien thuc pretrain trong vai buoc.
    _ftp = {id(p) for t in UPF for p in UPF[t].parameters()}
    rest = [p for m in (MOD, ETK, IMG) for p in m.parameters() if p.requires_grad]
    ftpp = [p for t in UPF for p in UPF[t].parameters() if p.requires_grad]
    opt = torch.optim.AdamW(
        [{'params': [p for p in rest if p.ndim > 1], 'weight_decay': WD, 'lr': LR},
         {'params': [p for p in rest if p.ndim <= 1], 'weight_decay': 0.0, 'lr': LR},
         # KHONG weight decay tren block pretrain. AdamW tach roi co tham so `w(1-lr*wd)` moi buoc;
         # voi LR/10 thi luc co (~0.4%/4k buoc) manh ngang gradient nhiem vu, nen `drift` do duoc
         # phan lon la BI BAO MON chu khong phai thich nghi — va do dung nhip `G_ft` di xuong.
         {'params': [p for p in ftpp if p.ndim > 1], 'weight_decay': 0.0, 'lr': LR/LR_FT_DIV},
         {'params': [p for p in ftpp if p.ndim <= 1], 'weight_decay': 0.0, 'lr': LR/LR_FT_DIV}])
    steps = math.ceil(len(I_TF) / (BATCH*ACCUM)) * EPOCHS; warm = int(WARMUP*steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    LOG.setdefault(sk, {})
    P_RES = os.path.join(CKPT, f'{sk}_resume.pt'); ep0 = 0
    if os.path.exists(P_RES):
        _r = torch.load(P_RES, map_location=dev, weights_only=False)
        _old = _r.get('arms')
        assert _old is None or list(_old) == list(ARMS), (
            f'checkpoint co {len(_old)} arm {list(_old)}, notebook co {len(ARMS)} arm.\n'
            f'  -> DOI RUN_TAG o C0 (vd "r2") roi chay lai tu C0. KHONG xoa gi.')
        MOD.load_state_dict(_r['mod']); ETK.load_state_dict(_r['etk'])
        IMG.load_state_dict(_r['img']); UPF.load_state_dict(_r['upf'])
        opt.load_state_dict(_r['opt']); sch.load_state_dict(_r['sch'])
        N_CLIP, N_SKIP, N_STEP, ep0 = _r['n_clip'], _r['n_skip'], _r['n_step'], _r['ep'] + 1
        print(f'{sk}: resume tu epoch {ep0+1}/{EPOCHS}')

    for ep in range(ep0, EPOCHS):
        # `UPF` chay EVAL y het ban goc (`backbone` luon o .eval()). No VAN nhan gradient va VAN
        # cap nhat — `eval()` chi tat dropout/drop-path. Neu de `.train()` thi `tpb_ft` co them
        # regularization NGAU NHIEN ma `tpb` khong co, va `G_ft` khong con do rieng "trong so thich
        # nghi" nua. C2 in ra cau hinh dropout de biet rui ro nay co that hay bang 0.
        MOD.train(); ETK.train(); IMG.train(); UPF.eval()
        t0, tot, nb = time.time(), {t: 0.0 for t in ARMS}, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            hl = shared_lo(b['image'].to(dev, non_blocking=True))
            hlp = shared_lo(b['image_prev'].to(dev, non_blocking=True))
            y = b['y'].to(dev); hs = b['has'].to(dev)
            pf, ppf = upper_frozen(hl), upper_frozen(hlp)      # MOT lan cho ca batch
            loss, bad = 0.0, []
            for t in ARMS:
                lg = arm_forward(t, pf, ppf, hl, hlp, b['erow'], hs)
                if not torch.isfinite(lg).all(): bad.append(t); N_SKIP[t] += 1; continue
                l_ = crit(lg.float(), y); loss = loss + l_; tot[t] += float(l_.detach())
            if bad and (sum(N_SKIP.values()) <= 3 or j % 400 == 0):
                print(f'   [{sk} ep{ep+1} buoc {j}] logit khong huu han: {bad}')
            if not torch.is_tensor(loss): opt.zero_grad(set_to_none=True); continue
            (loss / ACCUM).backward(); nb += 1
            if (j + 1) % ACCUM == 0:
                # Kiem gradient TRUOC khi buoc: total_norm = nan cho clip_coef = nan va BOI BAN
                # toan bo tham so -> mot buoc hong lam chet vinh vien.
                for t, ps in PARAM_OF.items():
                    gn = torch.nn.utils.clip_grad_norm_(ps, CLIP, error_if_nonfinite=False)
                    if not torch.isfinite(gn):
                        for p in ps:
                            if p.grad is not None: p.grad.zero_()
                        N_SKIP[t] += 1
                    elif float(gn) > CLIP: N_CLIP[t] += 1
                N_STEP += 1; opt.step(); sch.step(); opt.zero_grad(set_to_none=True)

        PV = predict(I_VA); PS = predict(I_SL); ensure_res()
        for t, p in PV.items(): np.save(os.path.join(RES57, f'p_{t}_{sk}_ep{ep+1}.npy'), p)
        for t, p in PS.items(): np.save(os.path.join(RES57, f'psl_{t}_{sk}_ep{ep+1}.npy'), p)
        if DO_TEST and ep >= EPOCHS - N_TEST_EP:
            PT = predict(I_TE); ensure_res()
            for t, p in PT.items(): np.save(os.path.join(RES57, f'pte_{t}_{sk}_ep{ep+1}.npy'), p)
            with open(P_TOUCH, 'a') as _f:
                _f.write(json.dumps(dict(nb='57', run=RUN_TAG, seed=seed, ep=ep+1, n=len(I_TE),
                                         arms=list(ARMS), muc_dich='theo doi',
                                         t=datetime.datetime.now().isoformat(timespec='seconds')))+'\n')
        with torch.no_grad():
            # TI LE BIEN DO THUC cua nhanh phim truoc: ‖phan du‖ / ‖patch‖. `pr_norm` chi la so
            # GIAN TIEP; day moi la dai luong vuot 1 thi sup. Do tren mot lo val co dinh.
            _bd = next(iter(loader(I_VA[:32], eval_tf, False, 32, nw=0)))
            _hb = shared_lo(_bd['image'].to(dev)); _hbp = shared_lo(_bd['image_prev'].to(dev))
            _hs2 = _bd['has'].to(dev)
            _pb, _ppb = upper_frozen(_hb), upper_frozen(_hbp)
            _ri = {}
            for t in IMG:
                _src = upper_ft(_hb, t) if ARMS[t]['ft'] else _pb
                _srp = upper_ft(_hbp, t) if ARMS[t]['ft'] else _ppb
                _ri[t] = float(((IMG[t](_src, _srp, _hs2) - _src).norm(dim=-1)
                                / (_src.norm(dim=-1) + 1e-9)).mean())
            _dr = {t: float(max((_first2d(UPF[t][k]) - _REF[i]).norm()
                                / (_REF[i].norm() + 1e-9) for k, i in enumerate(FT_BLOCKS)))
                   for t in UPF}
            # `drift` mot minh khong phan biet duoc BI CO LAI voi XOAY HUONG. `wnorm` = ‖w‖/‖w_goc‖:
            # < 1 la bi bao mon, ~1 ma drift van tang la thich nghi that.
            _wn = {t: float(np.mean([float(_first2d(UPF[t][k]).norm() / (_REF[i].norm() + 1e-9))
                                     for k, i in enumerate(FT_BLOCKS)])) for t in UPF}
        _e2 = {}
        for t, p in PV.items():
            ap = ap_cols(YM[VA], p)
            _e2[t] = dict(mAP_anh=float(np.nanmean(ap)), **tiers(ap),
                          sel=float(np.nanmean(ap_cols(YM[SL], PS[t]))),
                          pr_norm=float(IMG[t].pr.weight.norm()) if t in IMG else 0.0,
                          clip_pct=N_CLIP[t]/max(N_STEP, 1), drift=_dr.get(t, 0.0),
                          ratio_img=_ri.get(t, 0.0), wnorm=_wn.get(t, 0.0))
        LOG[sk][str(ep)] = _e2; json.dump(LOG, open(P_LOG, 'w'))
        torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                        upf=UPF.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(),
                        n_clip=N_CLIP, n_skip=N_SKIP, n_step=N_STEP, ep=ep, seed=seed,
                        arms=list(ARMS)), P_RES)
        g = lambda a, b: _e2[a]['mAP_anh'] - _e2[b]['mAP_anh']
        print(f'{sk} ep{ep+1:>2} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | '
              f'{(time.time()-t0)/60:.1f} ph | loss ' +
              ' '.join(f'{t}:{tot[t]/max(nb,1):.3f}' for t in ARMS))
        print('   VAL      ' + ' '.join(f'{t:>9}:{_e2[t]["mAP_anh"]:.4f}' for t in ARMS))
        print('   LAT CHON ' + ' '.join(f'{t:>9}:{_e2[t]["sel"]:.4f}' for t in ARMS))
        print('   TAIL     ' + ' '.join(f'{t:>9}:{_e2[t]["tail"]:.4f}' for t in ARMS))
        print('   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`')
        print(f'   G_ft = tpb_ft - tpb   {g("tpb_ft","tpb"):+.4f}   <- CAU HOI CHINH '
              f'(tail {_e2["tpb_ft"]["tail"]-_e2["tpb"]["tail"]:+.4f})')
        print('   ratio_img ' + ' '.join(f'{t}:{_ri[t]:.4f}' for t in IMG)
              + '   (vuot ~1 = phan du AT patch -> sup)')
        print('   pr_norm ' + ' '.join(f'{t}:{_e2[t]["pr_norm"]:.2f}' for t in ARMS if t in IMG)
              + f' | drift {_dr.get("tpb_ft", 0):.4f} wnorm {_wn.get("tpb_ft", 0):.4f}'
              + f' | clip {max(v["clip_pct"] for v in _e2.values()):.1%}')
        if max(N_SKIP.values()) > 0:
            print('   BUOC BI BO: ' + ' '.join(f'{t}:{v}' for t, v in N_SKIP.items() if v > 0))
    ensure_res()
    torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                    upf=UPF.state_dict(), seed=seed, arms=list(ARMS)),
               os.path.join(CKPT, f'{sk}.pt'))
    LOG[sk]['done'] = True; json.dump(LOG, open(P_LOG, 'w'))
    print(f'{sk}: xong')
print('tat ca seed xong')

## C7 · Kết quả + cổng

In [ ]:
LOG = json.load(open(P_LOG)); EPk = str(EPOCHS - 1)
R = pd.DataFrame([dict(seed=s, arm=t, **v) for s in SEEDS if EPk in LOG.get(f's{s}', {})
                  for t, v in LOG[f's{s}'][EPk].items()])
assert len(R), 'chua co ket qua nao'
SD = [s for s in SEEDS if EPk in LOG.get(f's{s}', {})]
piv = R.pivot_table(index='arm', values=['mAP_anh','head','medium','tail','sel','pr_norm','drift'],
                    aggfunc='mean').reindex(list(ARMS))
piv['bien_do'] = R.groupby('arm').mAP_anh.agg(lambda s: s.max()-s.min()).reindex(list(ARMS))
print(f'=== TUYET DOI (TB qua {len(SD)} seed) ===')
print(piv.to_string(float_format=lambda v: f'{v:.4f}'))

g  = lambda a, s: float(R[(R.arm == a) & (R.seed == s)].mAP_anh.iloc[0])
gt = lambda a, s: float(R[(R.arm == a) & (R.seed == s)]['tail'].iloc[0])
v  = [g('tpb_ft', s) - g('tpb', s) for s in SD]
vt = [gt('tpb_ft', s) - gt('tpb', s) for s in SD]
bd = (max(v) - min(v)) if len(v) > 1 else float('nan')
G_FT, T_FT = float(np.mean(v)), float(np.mean(vt))
print(f'\n=== CAU HOI CHINH ===')
print(f'  G_ft = tpb_ft - tpb   {G_FT:+.4f}  (bien do {bd:.4f}) | tail {T_FT:+.4f}')
print(f'  moc 53: t1_prior_both = 0.2560 | 56: tpb SUP con 0.2342 (tail 0.0685 vs 0.1077)')
print('\n' + '=' * 72)
# --- CONG SUP: doi chung phai KHOE thi hieu so moi doc duoc ---
# `56` day ra bai hoc dat: arm GIAU tham so on dinh hon, va cu sup uu tien danh vao doi chung
# NGHEO hon => moi hieu `arm - doi chung` bi THOI LEN. O `56`, `tpb_film_u - tpb = +0.0206` khong
# phai dung luong that: adapter, KE CA ban hang so, la mot phep AFFINE HOC DUOC dat ngay truoc
# `Tier1Img` — dung thu dap duoc cu `pr` chay qua da, ma `tpb` KHONG co.
# `53` cung dinh, chi khac vai: arm sup la doi chung `imgz`.
# `57` co DUNG hinh dang do: `tpb_ft` co 4 block trainable, `tpb` khong co gi.
TAIL_OK, PRN_OK = 0.090, 14.0      # 53: t1_prior_both tail 0.1077 / pr_norm 11.50 (binh thuong)
_tl = float(piv.loc['tpb', 'tail']); _pn = float(piv.loc['tpb', 'pr_norm'])
_sup = (_tl < TAIL_OK) or (_pn > PRN_OK)
print(f'cong SUP: tpb tail {_tl:.4f} (nguong {TAIL_OK}) | pr_norm {_pn:.2f} (nguong {PRN_OK})'
      f'  -> {"DOI CHUNG SUP" if _sup else "doi chung KHOE"}')
if _sup:
    print('  !! G_ft KHONG doc duoc: `tpb` sup, nen hieu so phan lon la "arm kia khong sup".')
    print('     Chay seed khac, HOAC sua bat on dinh cua Tier1Img truoc khi ket luan.')
print('CONG KHOA TRUOC (chi ap khi doi chung KHOE):')
print('  G_ft > +0.004  VA  tail(G_ft) >= -0.003   -> NHAN, di tiep')
print('  G_ft trong +0.002..+0.004                 -> vung xam, can seed thu ba')
print('  G_ft < +0.002                             -> DONG: dac trung 38a da du thang hang')
_v = ('KHONG DOC DUOC (doi chung sup)' if _sup else
      'NHAN' if (G_FT > 0.004 and T_FT >= -0.003) else
      'vung xam' if G_FT > 0.002 else 'DONG')
print(f'\n  -> {_v}')
print('=' * 72)
print('\n=== GIA THUYET PHU: mo bang co dap duoc cu sup cua Tier1Img khong ===')
for s in SD:
    _c = LOG[f's{s}']
    _pn = {t: [_c[str(e)][t]['pr_norm'] for e in range(EPOCHS) if str(e) in _c] for t in ('tpb','tpb_ft')}
    _tl = {t: [_c[str(e)][t]['tail'] for e in range(EPOCHS) if str(e) in _c] for t in ('tpb','tpb_ft')}
    print(f'  s{s} pr_norm  tpb {" ".join(f"{x:5.1f}" for x in _pn["tpb"])}')
    print(f'      pr_norm  ft  {" ".join(f"{x:5.1f}" for x in _pn["tpb_ft"])}')
    print(f'      tail     tpb {" ".join(f"{x:5.3f}" for x in _tl["tpb"])}')
    print(f'      tail     ft  {" ".join(f"{x:5.3f}" for x in _tl["tpb_ft"])}')
print('  53 do t1_prior_both pr_norm 11.50 / tail 0.1077 (binh thuong);')
print('  56 do tpb SUP: pr_norm 16.30 / tail 0.0685. Neu `ft` KHONG sup trong khi `tpb` sup thi')
print('  mo bang dap duoc bat on dinh — mot ket qua rieng, doc lap voi G_ft.')
ensure_res(); R.to_csv(os.path.join(RES57, 'main_table.csv'), index=False)